# Data Collection Portion of Project (API call)

### Importing all the necessary modules / libraries for this notebook and also most importantly calling our API key that we hid in "secret_key.env" file which is not in Github Repo for security purposes. 

In [ ]:
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

load_dotenv("secret_key.env")
FRED_API_KEY = os.getenv("FRED_API_KEY")

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
DATA_DIR.mkdir(exist_ok=True)
START = "1990-01-01"

# For Binder, we wont have an API Key so just read the dataset we already called from FRED API and saved in data folder
if FRED_API_KEY:
    from fredapi import Fred
    fred = Fred(api_key=FRED_API_KEY)
    USE_API = True
    print("FRED_API_KEY detected -- pulling live data from FRED.")
else:
    fred = None
    USE_API = False
    print("No FRED_API_KEY found -- loading cached CSVs from data/raw.")

### Getting all the series from FredAPI that we need for our project. All of the ones we call are straightdforward -- they all help gain more information to calculate the GDP Growth. Keeping in mind GDP = Consumer Spending + Investment + Govt. Spending + Net Exports, collected many columns that related to these 4 categories. Chose T10Y2Y since its known to be a leading indicator for economic turns. We also log to stabilize variance and ensure we have stationarity for future models we might implement.  

In [2]:
SERIES = {
    "INDPRO":       {"freq": "M", "tf": "log_diff", "label": "Industrial Production"},
    "RSAFS":        {"freq": "M", "tf": "log_diff", "label": "Retail Sales"},
    "PAYEMS":       {"freq": "M", "tf": "log_diff", "label": "Nonfarm Payrolls"},
    "UNRATE":       {"freq": "M", "tf": "diff",     "label": "Unemployment Rate"},
    "HOUST":        {"freq": "M", "tf": "log_diff", "label": "Housing Starts"},
    "PCE":          {"freq": "M", "tf": "log_diff", "label": "Personal Consumption"},
    "DGORDER":      {"freq": "M", "tf": "log_diff", "label": "Durable Goods Orders"},
    "ICSA":         {"freq": "W", "tf": "log_diff", "label": "Initial Claims"},
    "T10Y2Y":       {"freq": "D", "tf": "level",    "label": "10Y-2Y Spread"},
}
TARGET = "A191RL1Q225SBEA"
BENCHMARK = "GDPNOW"

### Pulls a Fred Series into an actual Series in our environment. Works to isolate the API Call so we maintain a simple codebase. 

In [ ]:
def fetch_series(code, start=START):
    if USE_API:
        s = fred.get_series(code, observation_start=start)
    else:
        # Read cached CSV written by a previous API run.
        path = RAW_DIR / f"{code}.csv"
        s = pd.read_csv(path, index_col=0, parse_dates=True).iloc[:, 0]
    s.name = code
    s.index = pd.to_datetime(s.index)
    return s.sort_index()

### Builds a dictonary of the series (raw data not the transformed version). Keeping it seperate from transformed in case we want to use different cases which doesn't use the transformed data (the logged version). 

In [5]:
def pull_all(series_map):
    return {code: fetch_series(code) for code in series_map}

raw = pull_all(SERIES)
target_raw = fetch_series(TARGET)
gdpnow_raw = fetch_series(BENCHMARK)

### Saving raw data in data/raw folder, helps so we can use it for offline if needed after first run. 

In [ ]:
def save_raw(data, target, bench):
    raw_dir = DATA_DIR / "raw"
    raw_dir.mkdir(exist_ok=True)
    for code, s in data.items():
        s.to_csv(raw_dir / f"{code}.csv")
    target.to_csv(raw_dir / f"{TARGET}.csv")
    bench.to_csv(raw_dir / f"{BENCHMARK}.csv")

# Only overwrite the cached CSVs when we actually pulled fresh data from FRED.
if USE_API:
    save_raw(raw, target_raw, gdpnow_raw)
else:
    print("Skipping save_raw -- using cached CSVs already on disk.")

### Summarizing to make sure all the columns are being pulled correctly, and there is a good amount of points for each column we can use. 

In [ ]:
def coverage_row(code, s):
    return {"code": code, "start": s.index.min().date(),
            "end": s.index.max().date(), "n": len(s)}

def coverage_table(data: dict, target: pd.Series, bench: pd.Series) -> pd.DataFrame:
    rows = [coverage_row(c, s) for c, s in data.items()]
    rows.append(coverage_row(TARGET, target))
    rows.append(coverage_row(BENCHMARK, bench))
    return pd.DataFrame(rows)

coverage_table(raw, target_raw, gdpnow_raw)

,code,start,end,n
0,INDPRO,1990-01-01,2026-03-01,435
1,RSAFS,1992-01-01,2026-02-01,410
2,PAYEMS,1990-01-01,2026-03-01,435
3,UNRATE,1990-01-01,2026-03-01,435
4,HOUST,1990-01-01,2026-01-01,433
5,PCE,1990-01-01,2026-02-01,434
6,DGORDER,1990-01-01,2026-02-01,434
7,ICSA,1990-01-06,2026-04-11,1893
8,T10Y2Y,1990-01-01,2026-04-17,9470
9,A191RL1Q225SBEA,1990-01-01,2025-10-01,144
